# Semantic IDs with RQ-VAE
Turn item embeddings into short tuples of discrete codes. Similar items get shared code prefixes.


In [ ]:
import sys, os; sys.path.append('..')
import numpy as np, torch, matplotlib.pyplot as plt
from config import cfg
from rqvae import RQVAE
from common import get_device, seed_everything
seed_everything(0); device = get_device()

In [ ]:
# real embeddings if you've run embed_items.py, else synthetic clusters
if os.path.exists('../data/item_emb.npy'):
    emb = np.load('../data/item_emb.npy')
else:
    rng = np.random.default_rng(0)
    K = 8; centers = rng.normal(size=(K, cfg.embed_dim))
    label = rng.integers(0, K, size=400)
    emb = centers[label] + 0.25*rng.normal(size=(400, cfg.embed_dim))
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
x = torch.tensor(emb, dtype=torch.float32, device=device)
x.shape

In [ ]:
# build the RQ-VAE and seed each codebook with k-means on the residual it sees
cfg.rq_levels = 3
model = RQVAE(cfg).to(device)
with torch.no_grad():
    r = model.encoder(x)
    for vq in model.quantizers:
        vq.kmeans_init(r); q,_,_,_ = vq(r); r = r - q
sum(p.numel() for p in model.parameters())

In [ ]:
# train: reconstruction + VQ loss (full-batch, it's tiny)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
rec_h, vq_h = [], []
for epoch in range(150):
    _, _, recon, vq = model(x)
    (recon + vq).backward(); opt.step(); opt.zero_grad()
    rec_h.append(recon.item()); vq_h.append(float(vq))
plt.plot(rec_h, label='reconstruction'); plt.plot(vq_h, label='vq')
plt.xlabel('epoch'); plt.legend(); plt.show()

In [ ]:
# every item is now a tuple of codes -> its Semantic ID
codes = model.encode_ids(x).cpu().numpy()
codes[:8]

Items that share the first code occupy the same region of embedding space.


In [ ]:
# project embeddings to 2D (PCA via SVD), color by the first Semantic ID code
xm = emb - emb.mean(0)
_, _, Vt = np.linalg.svd(xm, full_matrices=False)
proj = xm @ Vt[:2].T
plt.scatter(proj[:,0], proj[:,1], c=codes[:,0], cmap='tab20', s=12)
plt.title('color = first Semantic ID code'); plt.show()

In [ ]:
# residual quantization: the leftover shrinks at every level
with torch.no_grad():
    z = model.encoder(x[:1]); r = z; norms = [z.norm().item()]
    for vq in model.quantizers:
        q,_,_,_ = vq(r); r = r - q; norms.append(r.norm().item())
plt.bar(range(len(norms)), norms)
plt.xlabel('codes applied'); plt.ylabel('residual norm'); plt.show()